In [1]:
from dotenv import load_dotenv
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

load_dotenv(override=True)

True

In [2]:
# Định nghĩa State
class CustomerServiceState(TypedDict):
    customer_message: str
    category: str
    response: str

In [3]:
# Tạo các Node
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# Node 1: Phân loại yêu cầu
def classify_request(state: CustomerServiceState) -> CustomerServiceState:
    message = state["customer_message"]
    prompt = f"""Phân loại yêu cầu khách hàng sau vào một trong ba loại:
- "return": khách muốn đổi hoặc trả hàng
- "product_info": khách hỏi thông tin sản phẩm, giá, tồn kho
- "other": các yêu cầu khác

Chỉ trả về đúng một trong ba từ: return, product_info, hoặc other.

Yêu cầu của khách: {message}"""
    result = llm.invoke(prompt)
    category = result.content.strip().lower()

    return {"category": category}

# Node 2: Xử lý đổi/trả hàng
def handle_return(state: CustomerServiceState) -> CustomerServiceState:
    message = state["customer_message"]
    prompt = f"""Bạn là nhân viên CSKH của shop thời trang online.
Khách hàng muốn đổi hoặc trả hàng. Hãy trả lời thân thiện, 
hướng dẫn quy trình đổi trả trong 2-3 câu ngắn gọn.

Tin nhắn của khách: {message}"""
    result = llm.invoke(prompt)
    return {"response": result.content}

# Node 3: Tư vấn thông tin sản phẩm
def handle_product_info(state: CustomerServiceState) -> CustomerServiceState:
    message = state["customer_message"]
    prompt = f"""Bạn là nhân viên tư vấn của shop thời trang online.
Khách đang hỏi về sản phẩm. Hãy trả lời nhiệt tình, gợi ý thêm 
nếu phù hợp, trong 2-3 câu.

Tin nhắn của khách: {message}"""
    result = llm.invoke(prompt)
    return {"response": result.content}

# Node 4: Xử lý yêu cầu khác
def handle_other(state: CustomerServiceState) -> CustomerServiceState:
    message = state["customer_message"]
    prompt = f"""Bạn là nhân viên CSKH của shop thời trang online.
Khách có yêu cầu khác. Hãy trả lời lịch sự và hữu ích trong 2-3 câu.

Tin nhắn của khách: {message}"""
    result = llm.invoke(prompt)
    return {"response": result.content}

In [4]:
# Tạo Conditional Edge
def route_by_category(state: CustomerServiceState) -> str:
    category = state['category']

    if category == "return":
        return "handle_return"
    elif category == "product_info":
        return "handle_product_info"
    else:
        return "handle_other"

In [8]:
# Xây Graph và Compile
builder = StateGraph(CustomerServiceState)

builder.add_node("classify_request", classify_request)
builder.add_node("handle_return", handle_return)
builder.add_node("handle_product_info", handle_product_info)
builder.add_node("handle_other", handle_other)

builder.set_entry_point("classify_request")

builder.add_conditional_edges(
    "classify_request",
    route_by_category,
    {
        "handle_return": "handle_return",
        "handle_product_info": "handle_product_info",
        "handle_other": "handle_other"

    }
)

builder.add_edge("handle_return", END)
builder.add_edge("handle_product_info", END)
builder.add_edge("handle_other", END)

graph = builder.compile()

In [9]:
# test
test_messages = [
    "Em muốn đổi cái áo mua hôm qua vì size bị rộng quá",
    "Shop còn áo phông trắng size L không ạ?",
    "Shop mình có ship sang Mỹ không?"
]

for message in test_messages:
    print(f"\n{'='*50}")
    print(f"Khách: {message}")
    
    result = graph.invoke({
        "customer_message": message,
        "category": "",
        "response": ""
    })
    
    print(f"Phân loại: {result['category']}")
    print(f"Trả lời: {result['response']}")


Khách: Em muốn đổi cái áo mua hôm qua vì size bị rộng quá
Phân loại: return
Trả lời: Chào bạn! Cảm ơn bạn đã liên hệ với chúng tôi. Để đổi áo, bạn vui lòng gửi hàng trở lại cho chúng tôi kèm theo hóa đơn, và sau đó chúng tôi sẽ hỗ trợ bạn đổi sang size phù hợp. Nếu cần thêm thông tin, bạn hãy nhắn tin cho chúng tôi nhé!

Khách: Shop còn áo phông trắng size L không ạ?
Phân loại: product_info
Trả lời: Chào bạn! Hiện tại, shop còn áo phông trắng size L, bạn có thể xem thêm những mẫu khác để kết hợp nhé! Nếu bạn cần thêm thông tin hoặc gợi ý về trang phục, đừng ngần ngại hỏi nhé!

Khách: Shop mình có ship sang Mỹ không?
Phân loại: other
Trả lời: Chào bạn! Hiện tại shop chúng tôi chưa hỗ trợ giao hàng đến Mỹ. Tuy nhiên, nếu bạn có bất kỳ câu hỏi nào khác hoặc cần thêm thông tin, hãy cho chúng tôi biết nhé!
